# The shared finance corpus

Every finance case study on formalNLP argues from **one** corpus, generated by
`finance_corpus.py`. This notebook tours what it contains and runs the verification harness that
downstream topics rely on. Nothing here redefines the generator — every number comes from the module.

In [ ]:
import sys
from collections import Counter
from pathlib import Path

import numpy as np
from scipy.stats import spearmanr

sys.path.insert(0, str(Path.cwd()))
import finance_corpus as fc

docs = fc.generate_corpus(300, seed=0)
print(len(docs), "documents")
print("tones:", Counter(d.tone for d in docs))
print("doc types:", Counter(d.doc_type for d in docs))

## One document, with its gold annotations

Each document carries character-offset entity spans (`ORG`, `PERSON`, `MONEY`, `PERCENT`, `DATE`,
`PRODUCT`) and relations between them (`CEO_OF`, `REPORTED`, `ACQUIRED`, `GUIDED`) — the gold labels
the extraction topics evaluate against.

In [ ]:
d = docs[0]
print(d.doc_id, d.company, d.doc_type, "tone =", d.tone, "\n")
print(d.text, "\n")
for i, e in enumerate(d.entities):
    print(i, e.type, repr(d.text[e.start:e.end]))
for r in d.relations:
    print(d.text[d.entities[r.head].start:d.entities[r.head].end], r.rel,
          d.text[d.entities[r.tail].start:d.entities[r.tail].end])

## Rank–frequency structure

Content words are drawn from a Zipf distribution, so over its top 200 ranks the corpus approximates a
Zipfian rank–frequency profile: the least-squares slope of $\log f$ against $\log r$ sits near $-1$.
Its vocabulary is **closed** (a few hundred types from fixed word lists), so it is not a model of
vocabulary growth (Heaps' law) or of unseen events — topics that need those use their own data.

In [ ]:
counts = Counter(t for d in docs for t in fc._tokens(d.text))
freqs = np.array(sorted(counts.values(), reverse=True)[:200], dtype=float)
slope = np.polyfit(np.log(np.arange(1, 201)), np.log(freqs), 1)[0]
print(f"vocabulary: {len(counts)} types; Zipf slope over ranks 1–200: {slope:.3f}")
print("top 10:", counts.most_common(10))

## The stand-in sentiment lexicon

`STANDIN_LEXICON` is hand-written for this site — it is **not** the Loughran–McDonald dictionary,
whose license does not allow redistribution here. Its net score
$\text{score}(x) = (\#\text{pos} - \#\text{neg}) / \max(1, \#\text{tokens})$ tracks the gold tone,
but only moderately: documents mix sentence-level tones, hedge with uncertainty language in every
class, and sometimes carry polarity through negation ("were not strong") — exactly the cases a
bag-of-words lexicon misreads.

In [ ]:
for cat, words in fc.STANDIN_LEXICON.items():
    print(f"{cat:12s} {len(words):2d}  {sorted(words)[:6]} ...")
rho = spearmanr([fc.lexicon_score(d.text) for d in docs], [d.tone for d in docs]).statistic
print(f"\nSpearman(lexicon score, gold tone) = {rho:.3f}")

## Public EDGAR excerpts

In [ ]:
edgar = fc.load_edgar_excerpts()
print(len(edgar), "excerpts")
for e in edgar:
    print(f"- {e.doc_id} ({e.company}): {len(e.text.split())} words")

## Verification harness

In [ ]:
assert fc._run_harness() == 0